In [1]:
from huggingface_hub import hf_hub_download

CHECKPOINT_PATH = hf_hub_download(repo_id="KickItLikeShika/NileTTS-XTTS", filename="model.pth")
CONFIG_PATH = hf_hub_download(repo_id="KickItLikeShika/NileTTS-XTTS", filename="config.json")
VOCAB_PATH = hf_hub_download(repo_id="KickItLikeShika/NileTTS-XTTS", filename="vocab.json")

/home/ahmed.khaled/miniconda3/envs/xtts/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from huggingface_hub import hf_hub_download
import os
import pandas as pd

DATASET_REPO_ID = "KickItLikeShika/NileTTS-dataset"

# download only the CSV first
EVAL_CSV = hf_hub_download(
    repo_id=DATASET_REPO_ID,
    filename="metadata_eval.csv",
    repo_type="dataset"
)

df = pd.read_csv(EVAL_CSV, sep="|")

# a subset of first 10 audios
df_subset = df.head(10)

# Function to download audio on-demand
def get_audio_path(audio_filename):
    """Download and return local path to audio file"""
    return hf_hub_download(
        repo_id=DATASET_REPO_ID,
        filename=audio_filename,
        repo_type="dataset"
    )

# download just one audio file for testing
sample = df_subset.iloc[0]
audio_path = get_audio_path(sample['audio_file'])
print(f"Downloaded: {audio_path}")

Downloaded: /home/ahmed.khaled/.cache/huggingface/hub/datasets--KickItLikeShika--NileTTS-dataset/snapshots/4c2b2e0d07454796d8ebda5f761d4562d61c22cb/wavs/sales_34_00019.wav


In [3]:
import torch
import torchaudio
import pandas as pd
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

PLAYGROUND_OUTPUT_DIR = "playground_outputs"
os.makedirs(PLAYGROUND_OUTPUT_DIR, exist_ok=True)

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

cuda


In [4]:
# import sys
# if "XTTSv2-Finetuning-for-New-Languages" not in sys.path:
#     sys.path.insert(0, "XTTSv2-Finetuning-for-New-Languages")

In [5]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

config = XttsConfig()
config.load_json(CONFIG_PATH)

tts_model = Xtts.init_from_config(config)
tts_model.load_checkpoint(config, checkpoint_path=CHECKPOINT_PATH, vocab_path=VOCAB_PATH, use_deepspeed=False)
tts_model.to(device)
tts_model.eval()

In [6]:
eval_df = pd.read_csv(EVAL_CSV, sep='|')
print(f"len of eval dataset: {len(eval_df)}")

len of eval dataset: 950


In [7]:
eval_df

,audio_file,text,speaker_name
0,wavs/sales_34_00019.wav,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,SPEAKER_02
1,wavs/sales_26_00082.wav,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,SPEAKER_01
2,wavs/sales_15_00055.wav,لو التجربة كانت سيئة يعني اتعامل وحش. المنتج ط...,SPEAKER_01
3,wavs/medical_10_00012.wav,قاعدة اسمها 20-20-20 بالزبط وبسطتها دي هي سر ق...,SPEAKER_02
4,wavs/general_6_00008.wav,الطالب هنا فعليا اتحول من متلقي للعلم لمجرد رق...,SPEAKER_01
...,...,...,...
945,wavs/sales_31_00036.wav,يبقى لازم كل تعامل للعميل معايا يعكس السرعة دي...,SPEAKER_01
946,wavs/medical_5_00049.wav,والهوى بيحاول يعدي من ممر ضيط فبيعمل الصدر وال...,SPEAKER_02
947,wavs/general_52_00049.wav,لكنها بتلخص الواقع ده كله وهي القانون لا يحمي ...,SPEAKER_01
948,wavs/sales_26_00035.wav,ده بيسحب الفريق كله لتحت معاه. رابعاً الشخص ال...,SPEAKER_02


In [ ]:
# sample_idx = np.random.randint(0, len(eval_df))
sample_idx = 0
sample = df_subset.iloc[sample_idx]

text_to_synthesize = sample["text"]
ref_audio_relative_path = sample["audio_file"]
ref_audio_path = os.path.join(audio_path, ref_audio_relative_path)

print(f"original text (input to TTS): {text_to_synthesize}")
print(f"reference audio for speaker: {ref_audio_path}")

# synth audio
output_audio_filename = f"generated_sample_{sample_idx}.wav"
output_audio_path = os.path.join(PLAYGROUND_OUTPUT_DIR, output_audio_filename)

print(f"\synth audio to {output_audio_path}")
with torch.no_grad():
    # get conditioning latents from the reference audio
    gpt_cond_latent, speaker_embedding = tts_model.get_conditioning_latents(
        audio_path=ref_audio_path,
        gpt_cond_len=tts_model.config.gpt_cond_len,
        max_ref_length=tts_model.config.max_ref_len,
        sound_norm_refs=tts_model.config.sound_norm_refs,
    )
    wav = tts_model.inference(
        text=text_to_synthesize,
        language="ar",
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=0.7,
        length_penalty=1.0,
        repetition_penalty=5.0,
        top_k=50,
        top_p=0.8,
        do_sample=True,
        speed=1.0,
    )["wav"]

# save the generated audio waveform
# torchaudio.save(output_audio_path, torch.from_numpy(wav).unsqueeze(0).cpu(), 24000)
print("Audio generation complete.")

original text (input to TTS): فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل على نفسه و بيعمل حاجة واحدة بس. ده ممكن يبقى مشكلة بعدين. الأهم دلوقتي الشخص اللي عنده فضول.
reference audio for speaker: /home/ahmed.khaled/.cache/huggingface/hub/datasets--KickItLikeShika--NileTTS-dataset/snapshots/4c2b2e0d07454796d8ebda5f761d4562d61c22cb/wavs/sales_34_00019.wav
\synth audio to playground_outputs/generated_sample_0.wav


RuntimeError: Failed to create AudioDecoder for /home/ahmed.khaled/.cache/huggingface/hub/datasets--KickItLikeShika--NileTTS-dataset/snapshots/4c2b2e0d07454796d8ebda5f761d4562d61c22cb/wavs/sales_34_00019.wav: Could not open input file: /home/ahmed.khaled/.cache/huggingface/hub/datasets--KickItLikeShika--NileTTS-dataset/snapshots/4c2b2e0d07454796d8ebda5f761d4562d61c22cb/wavs/sales_34_00019.wav No such file or directory

In [11]:
from IPython.display import Audio
sample_rate = 24000 # XTTS uses 24kHz sample rate

Audio(data=wav, rate=sample_rate)